In [76]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader
from torch import nn
from torchvision.transforms import v2

In [91]:
train_dl = DataLoader(
    datasets.ImageFolder(
        root = "data/CIFAR10/train",
        transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale = True),
            v2.Normalize(
                mean=[0.4914, 0.4822, 0.4465],
                std=[0.2470, 0.2435, 0.2616]
            )
        ])
    ), batch_size = 64, shuffle = True
)

test_dl = DataLoader(
    datasets.ImageFolder(
        root = "data/CIFAR10/test",
        transform = v2.Compose([
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale = True),
                v2.Normalize(
                    mean=[0.4914, 0.4822, 0.4465],
                    std=[0.2470, 0.2435, 0.2616]
                    )
        ])
    ), batch_size = 64, shuffle = True 
)


In [78]:
print(train_dl.dataset.class_to_idx)
print(test_dl.dataset.class_to_idx)

{'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3, 'deer': 4, 'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}
{'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3, 'deer': 4, 'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}


In [100]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_linear = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(4096, 1024),
            nn.ReLU(),

            nn.Linear(1024, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )
    def forward(self, x):
        logits = self.conv_linear(x)
        return logits

In [101]:
def train_loop(model, loss_fn, optimizer, dataloader, device):
    model.train()
    batch_size = dataloader.batch_size
    size = len(dataloader.dataset)
    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        # forward 
        pred = model(x)
        loss = loss_fn(pred, y)

        # backward
        loss.backward()

        # grad descent
        optimizer.step()

        optimizer.zero_grad() # clear gradients

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(x)
            print(f"iteration {current}/{size}, loss = {loss}")

In [102]:
def test_loop(model, loss_fn, dataloader, device):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    total_loss, correct = 0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            pred = model(x)

            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        total_loss /= num_batches
        correct /= size

        print(f"test stats: avg loss: {total_loss}, accuracy = {correct * 100}") 


In [103]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [104]:
model = NeuralNetwork()

In [105]:
model.to(device)

NeuralNetwork(
  (conv_linear): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=4096, out_features=1024, bias=True)
    (8): ReLU()
    (9): Linear(in_features=1024, out_features=256, bias=True)
    (10): ReLU()
    (11): Linear(in_features=256, out_features=10, bias=True)
  )
)

In [106]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 5e-3, momentum=0.9)

epochs = 10

for epoch in range(epochs):
    print(f"epoch {epoch + 1}", "-" * 50, sep = "")
    train_loop(model, loss_fn, optimizer, train_dl, device)
    test_loop(model, loss_fn, test_dl, device)
print("Done!") # WOW that is sad 

epoch 1--------------------------------------------------
iteration 64/50000, loss = 2.296232223510742
iteration 6464/50000, loss = 2.0788416862487793
iteration 12864/50000, loss = 1.6453056335449219
iteration 19264/50000, loss = 1.4634912014007568
iteration 25664/50000, loss = 1.1886786222457886
iteration 32064/50000, loss = 1.1756025552749634
iteration 38464/50000, loss = 1.0794486999511719
iteration 44864/50000, loss = 1.3517450094223022
test stats: avg loss: 1.1317534617557647, accuracy = 59.35
epoch 2--------------------------------------------------
iteration 64/50000, loss = 1.058966040611267
iteration 6464/50000, loss = 0.896060585975647
iteration 12864/50000, loss = 1.2849199771881104
iteration 19264/50000, loss = 1.0687692165374756
iteration 25664/50000, loss = 0.8037587404251099
iteration 32064/50000, loss = 1.1406580209732056
iteration 38464/50000, loss = 1.0057445764541626
iteration 44864/50000, loss = 0.8381308913230896
test stats: avg loss: 0.9543314739397377, accuracy =